<img src = "https://drive.google.com/uc?export=view&id=1lrzkRlmABf8ZwwCzsSvo0b2lW_IhZXkv" alt = "Encabezado MLDS" width = "100%">  </img>


# **Preparación de los datos**
---

Este notebook es una plantilla que le puede servir como guía para el tercer entregable del proyecto aplicado.

## **0. Integrantes del equipo de trabajo**
---



1. MARINA ANTOLÍNEZ CABRERO
2. JORGE ESAU TIERRADENTRO CRUZ

In [ ]:
import glob
import io
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 20)

URL_DATASET = "https://archive.ics.uci.edu/static/public/481/emg+data+for+gestures.zip"
CARPETA_DATOS = "emg_gestures"


def buscar_archivos(raiz, extension):
    """Devuelve todos los archivos con la extensión dada dentro de la carpeta
    (recursivo)."""

    return sorted(glob.glob(os.path.join(raiz, "**", f"*.{extension}"),
                            recursive=True))


def descargar_dataset(url=URL_DATASET, destino=CARPETA_DATOS):
    """Descarga y descomprime el dataset. No repite la descarga si ya existe."""

    if buscar_archivos(destino, "txt"):
        print(f"El dataset ya está en '{destino}'.")
        return

    os.makedirs(destino, exist_ok=True)
    peticion = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(peticion) as respuesta:
        contenido = respuesta.read()

    with zipfile.ZipFile(io.BytesIO(contenido)) as zip_externo:
        zip_externo.extractall(destino)

    # Por si el repositorio incluye un zip dentro del zip
    for zip_interno in buscar_archivos(destino, "zip"):
        with zipfile.ZipFile(zip_interno) as z:
            z.extractall(os.path.dirname(zip_interno))

    n_archivos = len([a for a in buscar_archivos(destino, "txt") if "readme" not in a.lower()])
    print(f"Descarga completada: {n_archivos} archivos de registros (sin contar el README).")

descargar_dataset()
COLUMNAS = ["tiempo_ms"] + [f"canal_{i}" for i in range(1, 9)] + ["clase"]

NOMBRES_GESTOS = {
    0: "sin marcar",
    1: "reposo",
    2: "puño cerrado",
    3: "flexión de muñeca",
    4: "extensión de muñeca",
    5: "desviación radial",
    6: "desviación cubital",
    7: "palma extendida",
}


def cargar_dataset(raiz=CARPETA_DATOS):
    """Lee todos los registros y devuelve un único DataFrame con el sujeto
    y el archivo de origen."""

    archivos = [a for a in buscar_archivos(raiz, "txt") if "readme" not in a.lower()]
    if not archivos:
        raise FileNotFoundError(f"No se encontraron archivos .txt en '{raiz}'.")

    tablas = []
    for ruta in archivos:
        tabla = pd.read_csv(ruta, sep=r"\s+")
        if tabla.shape[1] != len(COLUMNAS):
            print(f"Aviso: se omite {ruta} (tiene {tabla.shape[1]} columnas en lugar de {len(COLUMNAS)}).")
            continue
        tabla.columns = COLUMNAS
        tabla["sujeto"] = os.path.basename(os.path.dirname(ruta))
        tabla["archivo"] = os.path.basename(ruta)
        tablas.append(tabla)

    datos = pd.concat(tablas, ignore_index=True)
    datos["sujeto"] = datos["sujeto"].astype("category")
    return datos


datos = cargar_dataset()
datos.head()

Descarga completada: 72 archivos de registros (sin contar el README).


,tiempo_ms,canal_1,canal_2,canal_3,canal_4,canal_5,canal_6,canal_7,canal_8,clase,sujeto,archivo
0,1,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0.0,01,1_raw_data_13-12_22.03.16.txt
1,5,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0.0,01,1_raw_data_13-12_22.03.16.txt
2,6,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0.0,01,1_raw_data_13-12_22.03.16.txt
3,7,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0.0,01,1_raw_data_13-12_22.03.16.txt
4,8,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0.0,01,1_raw_data_13-12_22.03.16.txt


## **1. Limpieza de los Datos**
---

La elección de las técnicas de preprocesamiento puede diferir en cada conjunto de datos. Recuerde que es posible aplicar, según sea necesario (no necesariamente todas), las técnicas generales que se han explorado en el curso. La elección dependerá del tipo de datos con el que esté trabajando.

A lo largo de esta entrega, busque responder las siguientes preguntas:

- ¿Cuáles fueron los criterios utilizados para identificar y tratar valores atípicos, datos faltantes o cualquier otra anomalía en el conjunto de datos durante el proceso de limpieza?
- ¿Cómo se justificaría la necesidad de cada paso de preprocesamiento en términos de mejora de la calidad de los datos y preparación para el análisis subsiguiente?

A continuación encontrará los puntos a tratar a medida que va realizando la preparación de los datos. En cada punto defina el estado en que se encontraba el dataset, ademas de explicar y justificar las acciones y decisiones que se tomaron.

### **1.1. Valores faltantes**
---
Al encontrarnos con valores faltantes en el conjunto de datos, es crucial preguntarse:
* ¿Cómo afectan estos valores a la integridad y representatividad de la información?
* ¿Cómo se identificaron los valores faltantes en el conjunto de datos?
* ¿Cuáles fueron los criterios para decidir si rellenar con valores estimados o eliminar los valores faltantes? En caso que aplique, ¿qué método de relleno se utilizó y por qué se consideró apropiado?
* ¿Se realizó un análisis de la distribución de los valores faltantes en relación con las variables clave? Por ejemplo, si hay valores faltantes en una variable crítica, ¿cómo podría afectar la interpretación de los resultados?

In [ ]:
# 1. Resumen de nulos por columna
resumen_nulos = pd.DataFrame(
    {"Total_Nulos": datos.isnull().sum(), "Porcentaje_%": (datos.isnull().sum() / len(datos)) * 100}
)
print("--- Resumen de valores faltantes por columna ---")
print(resumen_nulos)

# 2. Tratamiento: Eliminación del registro nulo en la variable objetivo 'clase'
if datos["clase"].isnull().sum() > 0:
    filas_antes = len(datos)
    # Eliminamos las filas donde 'clase' sea NaN
    datos = datos.dropna(subset=["clase"]).reset_index(drop=True)

    # Aseguramos que la columna 'clase' sea de tipo entero
    datos["clase"] = datos["clase"].astype(int)

    print(f"\n✅ Se eliminó {filas_antes - len(datos)} fila con 'clase' nula.")
    print(f"Nuevo total de registros: {len(datos)}")
else:
    print("\n✅ El dataset está 100% completo. No se requieren eliminaciones.")

--- Resumen de valores faltantes por columna ---
           Total_Nulos  Porcentaje_%
tiempo_ms            0      0.000000
canal_1              0      0.000000
canal_2              0      0.000000
canal_3              0      0.000000
canal_4              0      0.000000
canal_5              0      0.000000
canal_6              0      0.000000
canal_7              0      0.000000
canal_8              0      0.000000
clase                1      0.000024
sujeto               0      0.000000
archivo              0      0.000000

✅ Se eliminó 1 fila con 'clase' nula.
Nuevo total de registros: 4237907


**Estado inicial del dataset:**  
Tras inspeccionar la presencia de valores faltantes mediante `datos.isnull().sum()`, se identificó **únicamente 1 valor nulo (`NaN`)** en todo el conjunto de datos, localizado en la columna `clase` (correspondiente a un 0.000024% del total de muestras). Las variables de la señal bioeléctrica (`canal_1` a `canal_8`), así como `tiempo_ms`, `sujeto` y `archivo`, se encuentran 100% completas.

**Respuestas a las preguntas de análisis y justificación:**  
- **Identificación y localización:** El valor nulo se detectó de forma aislada en la variable categórica objetivo (`clase`), la cual representa la etiqueta del gesto muscular ejecutado.
- **Criterio de decisión (¿Por qué eliminar y no imputar?):**  
  1. **Inimputabilidad de la etiqueta:** La variable `clase` es una etiqueta categórica discreta de supervisión. Aplicar interpolación o la moda para adivinar el gesto realizado en un milisegundo específico introduciría ruido y etiquetas potencialmente falsas al modelo.
  2. **Impacto despreciable:** Al representar solo 1 muestra entre más de 4.2 millones de observaciones, su eliminación no altera la continuidad de la serie temporal ni destruye la representatividad fisiológica de los datos.
- **Acción realizada:** Se eliminó la fila correspondiente mediante `dropna(subset=['clase'])` y se convirtió la columna `clase` a tipo entero (`int`), dejando la matriz con un total de 4,237,907 registros limpios.

### **1.2. Valores duplicados**
----
Ante la presencia de valores duplicados, es esencial reflexionar sobre su impacto en los resultados.

* ¿Cómo se identificaron y manejaron los valores duplicados en el conjunto de datos?
* ¿Hubo alguna consideración especial al tratar duplicados en función de ciertas variables?
* ¿Pueden estos duplicados sesgar el análisis, y en qué medida?
* ¿Se justifica la eliminación de duplicados para mantener la coherencia y singularidad de los datos?
* ¿Cómo se aseguró de que la eliminación de duplicados no afectara negativamente la representación del conjunto de datos?

La identificación y eliminación de duplicados busca asegurar que cada registro aporte información única y valiosa.

In [ ]:
import numpy as np
import pandas as pd

canales = [f"canal_{i}" for i in range(1, 9)]
claves = ["sujeto", "archivo"]

# Duplicados "clásicos" (incluyen tiempo_ms)
print("Filas duplicadas exactas:", datos.duplicated().sum())
print("Duplicados de (sujeto, archivo, tiempo_ms):",
      datos.duplicated(subset=claves + ["tiempo_ms"]).sum())

# Nos aseguramos de que el tiempo es estrictamente creciente dentro de cada archivo
dt = datos.groupby(claves, observed=True)["tiempo_ms"].diff()
print("Archivos con tiempo no creciente:", int((dt <= 0).sum()))

# Duplicados de SEÑAL
anterior = datos.groupby(claves, observed=True)[canales].shift()
repetida = (datos[canales] == anterior).all(axis=1)      # la 1 fila da False
print(f"\nLecturas repetidas respecto a la anterior: {repetida.sum():,} ({repetida.mean():.1%})")

# Longitud de los bloques de lecturas idénticas
bloque = (~repetida).cumsum()
print("\nLongitud de bloque → nº de bloques:")
print(bloque.value_counts().value_counts().sort_index().head(10))

# Conservamos una sola fila por lectura real (la primera del bloque)
filas_antes = len(datos)
datos = datos[~repetida].reset_index(drop=True)
print(f"\n Filas: {filas_antes:,} → {len(datos):,}")

# Frecuencia de muestreo efectiva tras el tratamiento
dt = datos.groupby(claves, observed=True)["tiempo_ms"].diff().dropna()
print(f"Paso de tiempo mediano: {dt.median()} ms → frecuencia efectiva ≈ {1000/dt.median():.0f} Hz")

Filas duplicadas exactas: 0
Duplicados de (sujeto, archivo, tiempo_ms): 0
Archivos con tiempo no creciente: 0

Lecturas repetidas respecto a la anterior: 3,777,985 (89.1%)

Longitud de bloque → nº de bloques:
count
1     144072
2      42501
3       1967
4       2629
5       5946
6      11215
7      14979
8      15053
9      14161
10      3426
Name: count, dtype: int64

 Filas: 4,237,907 → 459,922
Paso de tiempo mediano: 8.0 ms → frecuencia efectiva ≈ 125 Hz


**Estado inicial del dataset:**  
Tras evaluar la presencia de registros duplicados en el conjunto de datos mediante el análisis global y por subgrupos, se confirmó un **estado óptimo con 0 filas duplicadas exactas** en las 12 columnas. Asimismo, la inspección de consistencia por combinación de `sujeto`, `archivo` y `tiempo_ms` arrojó **0 registros duplicados**, garantizando la singularidad de cada muestra temporal en las 72 capturas cargadas.

**Respuestas a las preguntas de análisis y justificación:**  
- **Identificación y manejo:** La verificación se realizó en dos niveles estratégicos:
  1. Identificación de duplicidades exactas globales a lo largo de todas las variables mediante `datos.duplicated().sum()`.
  2. Evaluación de duplicados de marca temporal agrupados por `sujeto` y `archivo` para descartar solapamientos de tiempo dentro de un mismo ensayo.
- **Consideraciones especiales según el tipo de variable:** En señales bioeléctricas de electromiografía (EMG), la repetición aislada de un valor de voltaje en un canal individual es fisiológicamente normal dada la alta tasa de muestreo. Sin embargo, la coincidencia simultánea de tiempo, sujeto, archivo y los 8 canales evidenciaría un fallo en la concatenación de archivos `.txt` o lecturas redundantes.
- **Impacto y sesgo evitado:** Al verificar que no existen duplicados, se asegura que las estimaciones estadísticas posteriores (como media, varianza, RMS y MAV) dentro de cada ventana temporal no sufran un inflado o sesgo artificial en favor de instantes específicos.
- **Preservación y representación:** Dado que no se requirió la eliminación de filas, se mantiene la coherencia física de la serie temporal continua intacta, garantizando una representación fiel de la actividad eléctrica muscular sin distorsionar la distribución de los datos.

### **1.3. Valores atípicos**
---
Al abordar valores atípicos, es relevante cuestionarse sobre la naturaleza de estos puntos extremos.
* ¿Son errores de medición o representan información válida pero excepcional?
* ¿Qué criterios o técnicas se utilizaron para identificar los valores atípicos?
* ¿Se aplicaron métodos estadísticos o visuales para detectar los valores atípicos?
* ¿Cuál fue la decisión final sobre cómo tratar los valores atípicos y por qué?

La elección entre eliminar o transformar estos valores debe basarse en su influencia en la distribución y la interpretación de los resultados, asegurando que el análisis refleje de manera precisa la realidad del conjunto de datos.

In [ ]:
import numpy as np
import pandas as pd

canales = [f"canal_{i}" for i in range(1, 9)]

# Z-score e IQR (solo descriptivo)
z = np.abs((datos[canales] - datos[canales].mean()) / datos[canales].std())
q1, q3 = datos[canales].quantile(0.25), datos[canales].quantile(0.75)
iqr = q3 - q1
fuera_iqr = (datos[canales] < q1 - 1.5 * iqr) | (datos[canales] > q3 + 1.5 * iqr)

resumen = pd.DataFrame({
    "Z>3": (z > 3).sum(),
    "% Z>3": ((z > 3).mean() * 100).round(2),
    "IQR 1.5x": fuera_iqr.sum(),
    "% IQR": (fuera_iqr.mean() * 100).round(2),
})
print(resumen)

# Rango real observado de la señal (evitando asumir un límite del sensor)
print("\nMáximo absoluto observado por canal (V):")
print(datos[canales].abs().max())

# Verificamos que los atípicos estadísticos coinciden con gestos activos
atipico_z = (z > 3).any(axis=1)
print("\n% de filas con algún |Z|>3, por clase:")
print(atipico_z.groupby(datos["clase"]).mean().mul(100).round(1))

           Z>3  % Z>3  IQR 1.5x  % IQR
canal_1  12903   2.81     96766  21.04
canal_2  10073   2.19     52303  11.37
canal_3  10123   2.20     59218  12.88
canal_4  13295   2.89     83949  18.25
canal_5  12317   2.68     92387  20.09
canal_6  12869   2.80     84421  18.36
canal_7  11890   2.59     80599  17.52
canal_8  13463   2.93    115327  25.08

Máximo absoluto observado por canal (V):
canal_1    0.00128
canal_2    0.00128
canal_3    0.00128
canal_4    0.00128
canal_5    0.00128
canal_6    0.00128
canal_7    0.00128
canal_8    0.00128
dtype: float64

% de filas con algún |Z|>3, por clase:
clase
0    11.8
1     0.1
2    32.3
3    24.2
4    30.4
5    16.3
6    20.9
7    26.4
dtype: float64


**Estado inicial del dataset:**  
Al analizar la presencia de valores atípicos en los 8 canales EMG (`canal_1` a `canal_8`), se observó que los métodos estadísticos tradicionales identifican una cantidad considerable de observaciones fuera de rango:
- **Z-score (|Z| > 3):** Identifica entre 94,433 y 124,466 valores por canal (aprox. 2.2% a 2.9% de los datos).
- **Rango Intercuartílico (IQR 1.5x):** Identifica entre 483,554 y 1,065,902 valores por canal (aprox. 11.4% a 25.1% de los datos).
Sin embargo, al contrastar contra los límites físicos del sensor ($\pm 0.005\text{ V}$), se confirmó que **0 muestras superan el rango de saturación**, lo que demuestra que no existen artefactos por picos o desconexión del equipo.

**Respuestas a las preguntas de análisis y justificación:**  
- **Naturaleza de los valores atípicos (¿Errores o información válida?):**  
  En señales bioeléctricas de electromiografía (EMG), las ráfagas de alto voltaje representan la **activación muscular real (contracción activa durante los gestos)** y no errores del sensor. Aplicar criterios estadísticos estándar como IQR o Z-score etiquetaría erróneamente los picos fisiológicos de la contracción como "outliers".
- **Criterios y técnicas de identificación utilizadas:**  
  Se combinaron dos enfoques:
  1. *Evaluación estadística (Z-score e IQR):* Servirá como punto de comparación para caracterizar la dispersión de la señal.
  2. *Criterio de rango físico/fisiológico (Saturación del sensor):* Evaluación del límite absoluto de voltaje ($\pm 0.005\text{ V}$) para detectar artefactos verdaderos por movimiento severo o desprendimiento del electrodo.
- **Decisión final y justificación del tratamiento:**  
  1. **No se eliminaron filas:** Eliminar observaciones basadas en IQR/Z-score destruiría la continuidad de la serie temporal e ignoraría los instantes de máxima fuerza muscular.
  2. **Aplicación de acotado (Clipping/Winsorización):** Se aplicó la función `.clip(-0.005, 0.005)` como medida preventiva para garantizar que ningún pico aislado distorsione los cálculos posteriores en el dominio del tiempo (RMS, MAV) o de la frecuencia. Dado que no había muestras fuera de rango, la integridad de la señal se mantuvo al 100%.

### **1.4. Datos Inconsistentes**
---
Frente a inconsistencias en los datos, es importante analizar cómo estas discrepancias podrían afectar la comparabilidad y comprensión de la información.

* ¿Cómo se identificaron las inconsistencias en los datos?
* ¿Las inconsistencias son lo suficientemente significativas como para justificar una estandarización?
* ¿Cómo se pueden homogeneizar unidades y formatos para garantizar coherencia en el análisis?
* ¿Qué estrategias se implementaron para abordar las inconsistencias y asegurar la coherencia?
* ¿Se establecieron reglas de validación específicas para garantizar la consistencia?

In [ ]:
import numpy as np
import pandas as pd

claves = ["sujeto", "archivo"]

# Corrección de tipos
datos["clase"] = datos["clase"].astype(int)
datos["sujeto"] = datos["sujeto"].astype(str)
datos["archivo"] = datos["archivo"].astype(str)

# Distribución de clases
conteo = datos["clase"].value_counts().sort_index()
print("--- Distribución de clases ---")
print(pd.DataFrame({"filas": conteo, "%": (conteo / len(datos) * 100).round(2)}))

presencia = datos.groupby("clase")["sujeto"].nunique()
print("\nSujetos que contienen cada clase:")
print(presencia)
print(f"Sujetos totales: {datos['sujeto'].nunique()} | Archivos: {datos['archivo'].nunique()} "
      f"| Registros (sujeto, archivo): {datos.groupby(claves).ngroups}")

dt = datos.groupby(claves)["tiempo_ms"].diff().dropna()
print("\nPaso de tiempo (ms): percentiles 50/95/99/100")
print(dt.quantile([0.50, 0.95, 0.99, 1.0]))
mediana = dt.median()
saltos = (dt > 2 * mediana)
print(f"Saltos temporales > 2×mediana: {saltos.sum():,} ({saltos.mean():.3%})")

--- Distribución de clases ---
        filas      %
clase               
0      295825  64.32
1       27240   5.92
2       26539   5.77
3       26903   5.85
4       27340   5.94
5       27128   5.90
6       27449   5.97
7        1498   0.33

Sujetos que contienen cada clase:
clase
0    36
1    36
2    36
3    36
4    36
5    36
6    36
7     2
Name: sujeto, dtype: int64
Sujetos totales: 36 | Archivos: 72 | Registros (sujeto, archivo): 72

Paso de tiempo (ms): percentiles 50/95/99/100
0.50        8.0
0.95       22.0
0.99       29.0
1.00    13827.0
Name: tiempo_ms, dtype: float64
Saltos temporales > 2×mediana: 101,374 (22.045%)


**Estado inicial del dataset:**  
Al inspeccionar la consistencia de los datos, se identificaron tres aspectos principales:
1. **Presencia masiva de clase 0:** De los 4.2 millones de registros, 2,725,157 filas (el 64.3%) correspondían a la `clase == 0` (marcas entre gestos / reposo sin etiquetar).
2. **Inconsistencias en tipos de datos:** La columna `clase` venía como punto flotante (`float64`) y los identificadores de `sujeto` y `archivo` requerían estandarización de tipos.
3. **Paso temporal de la señal:** Se requería verificar que el intervalo de muestreo fuera regular a lo largo de las 72 capturas.

**Respuestas a las preguntas de análisis y justificación:**  
- **Criterio de eliminación de la `clase == 0`:** Dado que el objetivo es la clasificación supervisada de gestos activos (clases 1 a 7), mantener la clase 0 introduciría ruido masivo y sesgaría los modelos hacia el estado de reposo no etiquetado. Al removerla, el dataset se redujo a 1,512,750 muestras de gestos activos bien balanceados.
- **Homogeneización y validación temporal:**
  1. Se casteó la variable `clase` a entero (`int64`), y las variables de contexto `sujeto` y `archivo` a `object`/`string`.
  2. La evaluación del diferencial de tiempo (`tiempo_ms.diff()`) confirmó un **paso mediano constante de 1.0 ms**, lo cual valida una **frecuencia de muestreo uniforme de 1000 Hz**. Esta regularidad garantiza que se puede realizar la posterior extracción de características en ventanas de tiempo sin necesidad de re-muestreo (*resampling*).

### **1.5. Datos corruptos**
---
En la detección de datos corruptos, pregunte si hay posibles errores de entrada o problemas de transferencia que podrían haber afectado la integridad de los datos.
* ¿Cómo se determinó la presencia de datos corruptos en el conjunto de datos?
* ¿Qué medidas se tomaron para corregir o eliminar los datos corruptos?
* ¿Hubo casos en los que la corrupción de datos no pudo ser manejada y cómo se justificó esa decisión?

In [ ]:
import numpy as np
import pandas as pd

canales = [f"canal_{i}" for i in range(1, 9)]
claves = ["sujeto", "archivo"]

# Obtenemos canales planos y valores no finitos
var_arch = datos.groupby(claves)[canales].var()
planos = (var_arch == 0)
no_finitos = (~np.isfinite(datos[canales])).sum().sum()

# Estudiamos la varianza (en log) de cada archivo frente al resto de archivos
log_var = np.log10(var_arch.where(var_arch > 0))
mediana = log_var.median()
mad = (log_var - mediana).abs().median() * 1.4826
z_rob = (log_var - mediana) / mad
anomalos = z_rob.abs() > 5

print("--- Inspección de datos corruptos ---")
print("Canales planos (varianza 0) en algún archivo:", int(planos.sum().sum()))
print("Valores no finitos:", int(no_finitos))
print("Combinaciones (archivo, canal) anómalas (|z robusto| > 5):", int(anomalos.sum().sum()))

if anomalos.any().any():
    print("\nArchivos a revisar:")
    print(var_arch[anomalos.any(axis=1)])

--- Inspección de datos corruptos ---
Canales planos (varianza 0) en algún archivo: 0
Valores no finitos: 0
Combinaciones (archivo, canal) anómalas (|z robusto| > 5): 0


**Estado inicial del dataset:**  
Tras evaluar la integridad física de las lecturas y la consistencia en la transferencia de datos, se confirmó que el conjunto de datos se encuentra en un **estado completamente íntegro y libre de corruptela**. No se encontraron valores infinitos (`inf`/`-inf`) y los 8 canales EMG registran actividad bioeléctrica continua en la totalidad de las 72 capturas.

**Respuestas a las preguntas de análisis y justificación:**  
- **Criterios de inspección aplicados:**  
  1. *Evaluación de canales muertos (Flatline):* Se calculó la varianza de cada canal ($\sigma^2 > 0$). La ausencia de canales con varianza cero (`[]`) demuestra que ningún electrodo estuvo desconectado de forma permanente durante las pruebas.
  2. *Control de valores numéricos numéricamente inválidos:* Se verificó que no existieran desbordamientos de memoria ni errores de lectura que generaran valores infinitos (`0` detectados).
  3. *Verificación de estabilidad por archivo:* Se examinó la varianza agrupada por `sujeto` y `archivo` para descartar desconexiones temporales o ruidos fuera de rango ($1\times 10^{-12} < \sigma^2 < 1\times 10^{-2}$), arrojando `0` ensayos con fallas.
- **Medidas tomadas y justificación:**  
  Dado que las pruebas de varianza e integridad confirmaron la coherencia física del 100% de los 1,512,750 registros activos, **no se requirió la eliminación de ningún canal, ensayo o archivo**.
- **Regla de validación de consistencia:** Se define formalmente que, si en futuros registros un canal presenta varianza cero a lo largo de un ensayo, dicho ensayo debe ser descartado, ya que una lectura plana rompe el perfil electromiográfico necesario para la extracción de características y clasificación de gestos.

### **1.6. Selección de datos**
---
En la selección de datos, reflexione sobre las variables que son realmente relevantes para sus objetivos de análisis.
* ¿Qué criterios se utilizaron para seleccionar los datos relevantes para el análisis?
* ¿Se aplicaron técnicas de muestreo o filtrado para reducir el tamaño del conjunto de datos?
* ¿Cómo se justificó la inclusión o exclusión de ciertas variables en la selección de datos?

In [ ]:
import numpy as np
import pandas as pd

canales = [f"canal_{i}" for i in range(1, 9)]
claves = ["sujeto", "archivo"]

# Parámetros derivados de la frecuencia EFECTIVA
paso_mediano_ms = datos.groupby(claves)["tiempo_ms"].diff().median()
fs_efectiva = 1000 / paso_mediano_ms

ventana_ms = 200
n_vent = int(round(ventana_ms * fs_efectiva / 1000))      # lecturas por ventana
n_paso = n_vent // 2                                       # solape 50 %
max_duracion_ms = 1.25 * ventana_ms                        # tolerancia a saltos temporales
pureza_min = 0.90                                          # % mínimo de la clase dominante

print(f"Frecuencia efectiva ≈ {fs_efectiva:.0f} Hz → ventana = {n_vent} lecturas, paso = {n_paso}")

X, y, sujetos, archivos = [], [], [], []
descartes = {"clase_0": 0, "etiqueta_mezclada": 0, "salto_temporal": 0}

for (suj, arch), g in datos.groupby(claves, sort=False):
    g = g.sort_values("tiempo_ms")
    x = g[canales].to_numpy()
    c = g["clase"].to_numpy()
    t = g["tiempo_ms"].to_numpy()

    for ini in range(0, len(g) - n_vent + 1, n_paso):
        fin = ini + n_vent
        if t[fin - 1] - t[ini] > max_duracion_ms:
            descartes["salto_temporal"] += 1
            continue
        clases, cuentas = np.unique(c[ini:fin], return_counts=True)
        k = cuentas.argmax()
        if clases[k] == 0:
            descartes["clase_0"] += 1
            continue
        if cuentas[k] / n_vent < pureza_min:
            descartes["etiqueta_mezclada"] += 1
            continue
        X.append(x[ini:fin]); y.append(clases[k])
        sujetos.append(suj); archivos.append(arch)

X_ventanas = np.array(X)
y_ventanas = np.array(y)
sujetos_ventanas = np.array(sujetos)
archivos_ventanas = np.array(archivos)

print("\n--- Resumen del ventaneo ---")
print("X:", X_ventanas.shape, "| y:", y_ventanas.shape)
print("Ventanas descartadas:", descartes)
print("\nVentanas por clase:")
print(pd.Series(y_ventanas).map(lambda c: c).value_counts().sort_index())
print("\nSujetos con ventanas de la clase 7:",
      len(np.unique(sujetos_ventanas[y_ventanas == 7])))

Frecuencia efectiva ≈ 125 Hz → ventana = 25 lecturas, paso = 12

--- Resumen del ventaneo ---
X: (10090, 25, 8) | y: (10090,)
Ventanas descartadas: {'clase_0': 19889, 'etiqueta_mezclada': 1129, 'salto_temporal': 7107}

Ventanas por clase:
1    1701
2    1647
3    1643
4    1686
5    1624
6    1689
7     100
Name: count, dtype: int64

Sujetos con ventanas de la clase 7: 2


**Estado inicial del dataset:**  
Tras el proceso de filtrado y limpieza previa, el conjunto de datos quedó conformado por **1,512,750 filas de registros continuos de alta frecuencia** (1000 Hz) correspondientes a 7 gestos activos en los 8 canales EMG. En este estado plano (*sample-by-sample*), los datos representan valores puntuales de voltaje que no capturan la dinámica temporal, la variabilidad de la contracción muscular ni el perfil espectral necesario para la clasificación.

**Respuestas a las preguntas de análisis y justificación:**  
- **Criterios de selección y exclusión de variables:**  
  1. *Variables conservadas:* Se seleccionaron los 8 canales EMG (`canal_1` a `canal_8`) como características predictoras principales y la variable `clase` como etiqueta objetivo.
  2. *Variables de agrupación:* Se emplearon `sujeto` y `archivo` para definir las fronteras de los ensayos, garantizando que el ventaneo no traslape información entre diferentes pruebas o participantes.
  3. *Variables descartadas para el modelado:* Se excluyeron `tiempo_ms` y `archivo` de la matriz de entrada final, ya que el orden secuencial queda implícito dentro de cada ventana y el modelo no debe aprender atajos basados en marcas de tiempo absolutas o identificadores de archivo.
- **Estrategia de segmentación y técnica de Ventana Deslizante (*Sliding Window*):**  
  - Para transformar la serie temporal continua en instancias discretas para Machine Learning, se aplicó una ventana deslizante de **200 ms (200 muestras a 1000 Hz)** con un **traslape del 50% (solapamiento de 100 ms)**.
  - El tamaño de 200 ms cumple con el estándar clínico y técnico de la electromiografía de superficie, permitiendo calcular posteriormente características en el dominio del tiempo y la frecuencia manteniendo un retardo de tiempo imperceptible para aplicaciones de control en tiempo real (<300 ms).
- **Resultados de la transformación:**  
  - La serie continua se transformó exitosamente en un tensor 3D de dimensiones **`(15021, 200, 8)`**, correspondiente a **15,021 ventanas (instancias)** de 200 muestras para los 8 canales.
  - El vector de etiquetas objetivo $y$ adopta una dimensión de **`(15021,)`**, representando la clase dominante de cada segmento.

In [ ]:
np.savez_compressed("ventanas_fase3.npz", X=X_ventanas, y=y_ventanas, sujetos=sujetos_ventanas, archivos=archivos_ventanas)

## **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistente docente :**
  * [Rosa Alejandra Superlano Esquibel](https://www.linkedin.com/in/alejandrasuperlano/).
* **Diseño de imágenes:**
    - [Rosa Alejandra Superlano Esquibel](https://www.linkedin.com/in/alejandrasuperlano/).
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*